In [ ]:
# ============================================================================
# 0) 导入依赖
# ============================================================================
from datetime import datetime
from pathlib import Path
from functools import partial

import polars as pl
from vnpy.trader.constant import Interval, Exchange, FactorType
from vnpy.trader.object import FactorRequest
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha.dataset import AlphaDataset
from vnpy.alpha.dataset.processor import (
    process_drop_na,
    process_fill_na,
    process_cs_norm,
)
from vnpy.factor_define import FACTOR_NAMES


In [ ]:
# ============================================================================
# 1) 全局参数配置
# ============================================================================

# -- 指数与路径 --
VT_INDEX_SYMBOL: str = "000300.SSE"
BASE_PATH: Path = Path("D:/Aquant project/MF")
LAB_PATH: Path = BASE_PATH / "MF_lab"

# -- 时间跨度 --
# 全量数据起止（用于加载K线和因子）
START: datetime = datetime(2018, 1, 1)
END: datetime = datetime(2026, 5, 8)

# 训练 / 验证 / 回测 三段切分
TRAIN_START, TRAIN_END = datetime(2018, 1, 1), datetime(2023, 12, 31)
VALID_START, VALID_END = datetime(2024, 1, 1), datetime(2024, 12, 31)
TEST_START,  TEST_END  = datetime(2025, 1, 1), END

# -- 因子参数 --
# 滞后天数（0 表示只用当期，不生成 lag 特征）
LAG_DAYS: int = 0

# 标签：future_days 日后的开盘收益率
# 周一开盘买入 -> 下周一开盘卖出，跨 5 个交易日
FUTURE_DAYS: int = 6

# -- 因子列表 --
# 使用 factor_define 中注册的全部因子，或手动指定子集
FACTORS: list[str] = FACTOR_NAMES
# FACTORS = ["streverse_1m"]  # <-- 调试单因子时取消注释

print(f"因子数量: {len(FACTORS)}, 滞后天数: {LAG_DAYS}, 未来期数: {FUTURE_DAYS}")

In [ ]:
# ============================================================================
# 2) 初始化 Lab & 加载股票池
# ============================================================================

lab = AlphaLab(str(LAB_PATH))

# 加载指定时间范围内的沪深300成分股列表
component_symbols = lab.load_component_symbols(VT_INDEX_SYMBOL, START, END)
print(f"成分股数量: {len(component_symbols)}")

In [ ]:
# ============================================================================
# 3) 加载 K 线并创建 AlphaDataset
# ============================================================================

# 加载日线无复权数据
df = lab.load_bar_df(
    vt_symbols=component_symbols,
    interval=Interval.DAILY,
    start=START,
    end=END,
    extended_days=0,
    adjust_type="none",
)

# 只保留 datetime + vt_symbol + open（标签计算需要 open）
daily_open = df.select(["datetime", "vt_symbol", "open"]).sort(["vt_symbol", "datetime"])

# 释放原始大表，减少内存峰值
del df

dataset = AlphaDataset(
    df=daily_open,
    train_period=(TRAIN_START, TRAIN_END),
    valid_period=(VALID_START, VALID_END),
    test_period=(TEST_START, TEST_END),
    process_type="append",
)

print(f"Dataset 创建完成 | daily_open shape: {daily_open.shape}")

In [ ]:
# ============================================================================
# 4) 流式加载因子并生成滞后特征
# ============================================================================

symbols = []
exchanges = []
for vt_symbol in component_symbols:
    symbol, exchange = vt_symbol.split('.')
    symbols.append(symbol)
    exchanges.append(Exchange(exchange))

print(f"开始处理 {len(FACTORS)} 个因子，每个生成 {LAG_DAYS} 个滞后特征...\n")

for i, factor_name in enumerate(FACTORS, 1):
    # 构造因子请求（Price and Volume 类型）
    req = FactorRequest(
        is_FD=True,
        start=START,
        end=END,
        symbols=symbols,
        exchanges=exchanges,
        factor_name=factor_name,
        factor_type=FactorType.PRICE_AND_VOLUME,
    )

    factor_df = lab.load_factor(req)

    # 使用 AlphaDataset.add_lag_features 向量化生成滞后
    # lag=0 时特征名 = factor_name，lag>0 时 = factor_name_lag_{lag}
    print(f"  [{i}/{len(FACTORS)}] {factor_name} ")
    dataset.add_lag_features(factor_name, factor_df, lag_days=LAG_DAYS)


print(f"\n✓ 因子加载完成，共 {len(dataset.feature_results)} 个特征结果")

In [ ]:
# ============================================================================
# 5) 设置标签：未来收益率
# ============================================================================

# 用 open 价格计算 future_days 后的收益率
# ts_delay(open, -n) 表示向前取 n 期的 open
label_expr = f"(ts_delay(open, -{FUTURE_DAYS}) / ts_delay(open, -1)) - 1"
dataset.set_label(label_expr)

print(f"标签设置完成: {label_expr}")

In [ ]:
# ============================================================================
# 6) 准备数据：计算特征表达式 + 合并结果 + 成分股过滤
# ============================================================================

# 加载成分股过滤器（确保每天只保留在指数内的股票）
filters = lab.load_component_filters2(VT_INDEX_SYMBOL, START, END)

# 计算表达式特征、合并 feature_results、过滤成分股
# prepare_data 执行后会自动释放 self.df 和 self.feature_results，降低内存峰值
dataset.prepare_data(filters=filters)

print("prepare_data 完成")

In [ ]:
# ============================================================================
# 7) 【可选】快速检查数据状态
# ============================================================================

# 查看 learn_df 的 shape 和列名（首次访问会触发懒加载或读取内存对象）
if dataset.learn_df is not None:
    print(f"learn_df shape: {dataset.learn_df.shape}")
    print(f"columns: {dataset.learn_df.columns[:10]}...")  # 前10列

In [ ]:
# ============================================================================
# 8) 添加数据预处理器
# ============================================================================

# 获取特征列名（排除 datetime / vt_symbol / label）
_feature_cols = [
    c for c in dataset.learn_df.columns
    if c not in ("datetime", "vt_symbol", "label")
]

# # -- 学习数据预处理器 --
# # 1) 删除含缺失值的行
# dataset.add_processor("learn", partial(process_drop_na))

# # 2) 截面 z-score 标准化（特征 + 标签）
# dataset.add_processor(
#     "learn",
#     partial(process_cs_norm, names=_feature_cols + ["label"], method="zscore")
# )
#
# # -- 推理数据预处理器 --
# # 1) 截面 z-score 标准化（特征 + 标签，与 learn 保持一致）
# dataset.add_processor(
#     "infer",
#     partial(process_cs_norm, names=_feature_cols + ["label"], method="zscore")
# )
#
# # 2) 填充缺失值（推理时不允许有 null）
# dataset.add_processor("infer", partial(process_fill_na, fill_value=0))

print(f"预处理器添加完成 | 特征列数: {len(_feature_cols)}")

In [ ]:
# ============================================================================
# 9) 执行预处理
# ============================================================================

dataset.process_data()
print("process_data 完成")

In [ ]:
# ============================================================================
# 10) 保存 Dataset（Parquet + JSON 格式，内存友好）
# ============================================================================

DATASET_NAME = "v100"
lab.save_dataset(DATASET_NAME, dataset)

print(f"Dataset 已保存: {DATASET_NAME}")

In [ ]:
# ============================================================================
# 11) 【可选】加载验证
# ============================================================================

# 重新加载，验证懒加载是否正常
ds_check = lab.load_dataset(DATASET_NAME)
print(f"加载完成，类型: {type(ds_check).__name__}")
print(f"learn_df shape: {ds_check.learn_df.shape}")
print(f"TRAIN segment shape: {ds_check.fetch_learn(list(ds_check.data_periods.keys())[0]).shape}")